# Quy tắc làm sạch dùng chung

Quy tắc làm sạch dùng chung cho tập train và tập test HCM_3.

Mỗi hàm bước (B1 … B9) nhận bảng làm việc `d` và báo cáo `bc`, sửa `d` tại chỗ.
Riêng hai bước khử trùng lặp trả về bảng mới vì số dòng thay đổi.

Notebook này chỉ định nghĩa hằng số và hàm, không đọc dữ liệu. Hai notebook pipeline nạp nó bằng `%run`.

In [ ]:
import csv
import re
import unicodedata

import numpy as np
import pandas as pd

## Tên cột

In [ ]:
PHUONG, DUONG, DOAN, CT = 'Xã/Phường/Thị trấn mới', 'Đường phố', 'Đoạn đường', 'Chi tiết'
KC, VT = 'Khoảng cách đến đường chính (m)', 'Vị trí trong khung giá'
NGO = 'Độ rộng ngõ/ngách nhỏ nhất (Từ đường chính đến BĐS)'
DT, MT, CD = 'Diện tích (m2)', 'Kích thước mặt tiền (m)', 'Kích thước chiều dài'
SMT, HD, LTKD, YTK = 'Số mặt tiền tiếp giáp', 'Hình dáng', 'Lợi thế kinh doanh', 'Yếu tố khác'
QH, MA_TS = 'Thành phố/Quận/Huyện/Thị xã', 'Mã tài sản'
NGAY = 'Thời điểm hiệu lực'
TARGET = 'Đơn giá quyền sử dụng đất (đ/m2)1'
CS_NGO, CS_DT, CS_HD = 'Chi_so_ngo', 'Chi_so_dientich', 'Chi_so_hinhdang'

CLEAN = [PHUONG, DUONG, DOAN, CT, KC, VT, NGO, DT, MT, CD, SMT, HD, LTKD, YTK]                         # 14 cột làm sạch
KEEP = ['Ghep', 'Don_gia_TB', 'Min', 'Max', 'Độ lệch chuẩn', 'Số lượng mẫu', 'Tháng', 'Quý_Năm', 'Vùng']  # 9 cột giữ nội dung
THEM = [QH, MA_TS]                                                                                 # 2 cột bổ sung
KHOA_TRUNG = CLEAN + [TARGET]                                                                      # khoá khử trùng lặp B2
KHOA_B7 = [c if c != CT else '_chi_tiet_b1' for c in KHOA_TRUNG]                                   # B7: Chi tiết trước khi tách
COT_MOI = ['hinh_dang_sach', CS_NGO, CS_DT, CS_HD, 'ty_le_kt_dt', 'ty_le_dai_rong',
           'la_hem', 'co_mat_tien', 'chi_la_mat_tien', 'la_so_nha', 'co_bang_chung_mat_tien',
           'da_sua_vt_theo_mat_tien', 'kc_thieu_du_lieu', 'da_sua_kich_thuoc', 'canh_bao_vt1', 'canh_bao_mt_vt234']
COT_DOC = CLEAN + KEEP + THEM + [TARGET, NGAY, CS_NGO, CS_DT, CS_HD]                               # cột cần đọc từ file gốc

VT234 = ['vt2', 'vt3', 'vt4']
NGAY_BANG_HIEN_HANH = pd.Timestamp('2026-01-01')   # bảng hệ số hình dáng của công ty đổi từ ngày này
HE_SO_MAC_DINH = 0.95                              # hệ số hình dáng khi không tra được

## Báo cáo và kiểm tra

In [ ]:
class BaoCao:
    """Ghi số dòng từng rule tác động và so số đo với kỳ vọng.

    ky_vong: {tên: giá trị} đo trên dữ liệu gốc; tên không có trong bảng thì chỉ in ra.
    strict:  True thì dừng ngay khi lệch kỳ vọng, False thì chỉ in cảnh báo.
    """

    def __init__(self, ky_vong=None, strict=True):
        self.ky_vong = ky_vong or {}
        self.strict = strict
        self.dong = []

    def ghi(self, ma_rule, so_dong, cot):
        """Ghi một rule vào báo cáo, in ra và so với kỳ vọng nếu có."""
        so_dong = int(so_dong)
        self.dong.append({'ma_rule': ma_rule, 'so_dong_tac_dong': so_dong, 'cot_bi_sua': cot})
        print(f'  [{ma_rule}] {so_dong:>6} dòng · {cot}')
        self._so_sanh(ma_rule, so_dong)

    def kiem(self, ten, thuc_te):
        """So một số đo với kỳ vọng; không có kỳ vọng thì in ra."""
        thuc_te = thuc_te.item() if hasattr(thuc_te, 'item') else thuc_te
        if ten not in self.ky_vong:
            print(f'  {ten} = {thuc_te!r}')
        self._so_sanh(ten, thuc_te)

    def _so_sanh(self, ten, thuc_te):
        kv = self.ky_vong.get(ten)
        if kv is None or thuc_te == kv:
            return
        msg = f'{ten}: kỳ vọng {kv!r}, thực tế {thuc_te!r}'
        if self.strict:
            raise AssertionError(msg)
        print('  !! ' + msg)

    def luu(self, path):
        pd.DataFrame(self.dong).to_csv(path, index=False, encoding='utf-8-sig')


def dk(ten, dieu_kien):
    """Kiểm thử đúng/sai; sai thì dừng."""
    if not dieu_kien:
        raise AssertionError('Kiểm thử thất bại: ' + ten)
    print('  OK', ten)


def so_dong_khac(truoc, sau):
    """Số ô khác nhau giữa hai Series (NaN coi như chuỗi rỗng)."""
    return int((truoc.fillna('') != sau.fillna('')).sum())

## B0 · Đọc dữ liệu và chọn cột

In [ ]:
def doc_file_goc(path):
    """Đọc file Excel gốc, chỉ lấy các cột pipeline cần (cột không có trong file thì bỏ qua)."""
    can = set(COT_DOC)
    return pd.read_excel(path, usecols=lambda c: c in can)


def tao_bang_lam_viec(raw):
    """Chọn cột và thêm các cột phụ (tên bắt đầu bằng `_`, bị bỏ trước khi xuất)."""
    thieu = [c for c in KEEP + THEM + [TARGET] if c not in raw.columns]
    if thieu:
        print('Cột không có trong file này, bỏ qua:', thieu)
    d = raw[CLEAN + [c for c in KEEP + THEM if c in raw.columns] + [TARGET]].copy()
    d['_dong_goc'] = np.arange(len(d)) + 2                  # dòng 1 của Excel là tiêu đề
    d['_ngay_hieu_luc'] = doc_ngay(raw[NGAY])
    d['_do_rong_goc'] = raw[NGO]                            # giá trị gốc cho B8
    d['_dien_tich_goc'] = raw[DT]
    d['_hinh_dang_nguyen_van'] = raw[HD]
    d['_cs_ngo_cty'] = raw[CS_NGO]                          # chỉ số của công ty, chỉ để đối chiếu
    d['_cs_dt_cty'] = raw[CS_DT]
    d['_cs_hd_cty'] = raw[CS_HD].round(3)
    return d


def doc_ngay(s):
    return pd.to_datetime(s, format='%d/%m/%Y', errors='coerce')


def cot_xuat(d):
    """Các cột của file đầu ra, theo thứ tự cố định."""
    return [c for c in CLEAN + KEEP + THEM + [TARGET] + COT_MOI if c in d.columns]

## B1 · Chuẩn hoá text

In [ ]:
JUNK = {'-', '--', '.', '*', '_', '/', 'n/a', 'null', '#n/a'}


def chuan_hoa_text(s):
    if pd.isna(s):
        return np.nan
    s = unicodedata.normalize('NFKC', str(s))                    # thống nhất cách mã hoá ký tự
    s = re.sub(r'[​-‏﻿ ⁠]', '', s)      # ký tự vô hình
    s = re.sub(r'\s+', ' ', s).strip().lower()                   # gộp khoảng trắng, viết thường
    return np.nan if (s == '' or s in JUNK) else s


def cac_cot_chu(d):
    """Cột chữ = không phải số, không phải ngày (pandas 3 dùng dtype `str` thay vì `object`)."""
    return [c for c in d.columns
            if not c.startswith('_')
            and not pd.api.types.is_numeric_dtype(d[c])
            and not pd.api.types.is_datetime64_any_dtype(d[c])]


def b1_chuan_hoa_text(d, bc):
    cols = cac_cot_chu(d)
    bc.kiem('B1_so_cot_chu', len(cols))
    for c in cols:
        d[c] = d[c].map(chuan_hoa_text)

## B2, B7 · Khử trùng lặp

In [ ]:
def khu_trung_lap(d, khoa, buoc):
    """Giữ bản có ngày hiệu lực muộn nhất của mỗi nhóm trùng; hoà thì giữ dòng đứng trước trong file gốc.

    Trả về (bảng đã khử trùng, các dòng bị loại kèm `ma_ban_giu` là Mã tài sản của bản được giữ).
    """
    tmp = d.sort_values(['_ngay_hieu_luc', '_dong_goc'], ascending=[False, True])   # mới nhất lên đầu
    nhom = tmp.groupby(khoa, dropna=False).ngroup()
    la_ban_giu = ~nhom.duplicated()
    ma_ban_giu = tmp.loc[la_ban_giu, MA_TS].set_axis(nhom[la_ban_giu])
    bi_loai = tmp[~la_ban_giu].assign(buoc_loai=buoc, ma_ban_giu=nhom[~la_ban_giu].map(ma_ban_giu))
    giu = tmp[la_ban_giu].sort_values('_dong_goc').reset_index(drop=True)
    return giu, bi_loai

## B4.1 · Cột số và Vị trí trong khung giá

In [ ]:
def b4_cot_so(d, bc):
    vt5 = d[VT] == 'vt5'
    d.loc[vt5, VT] = 'vt4'
    bc.ghi('B4_vt5_gop_vt4', vt5.sum(), VT)

    for ma, mask, cot in [('B4_smt_nan', d[SMT].eq(0) | d[SMT].gt(4), SMT),
                          ('B4_do_rong_0_nan', d[NGO].eq(0), NGO),
                          ('B4_mt_0_nan', d[MT].eq(0), MT),
                          ('B4_kc_qua_lon_nan', d[KC].gt(2500), KC)]:
        bc.ghi(ma, mask.sum(), cot)
        d.loc[mask, cot] = np.nan

## B4.2 · Quận/Huyện

In [ ]:
def b4_quan_huyen(d, bc):
    m = d[QH] == 'thị xã thuận an'
    d.loc[m, QH] = 'thành phố thuận an'
    bc.ghi('B4_quan_huyen_doi_ten', m.sum(), QH)
    bc.kiem('B4_quan_huyen_so_gia_tri', d[QH].nunique())

## B4.3 · Đoạn đường

In [ ]:
def chuan_doan_duong(s):
    if pd.isna(s):
        return s
    s = re.sub(r'^(?:đầu đường\s*-\s*cuối đường|toàn tuyến)$', 'trọn đường', s)
    s = re.sub(r'^(bề rộng mặt đường) dưới\b', r'\1 nhỏ hơn', s)
    s = re.sub(r'(\d)\s+m\b', r'\1m', s)
    s = re.sub(r'\.$', '', s)
    return s


def b4_doan_duong(d, bc):
    truoc = d[DOAN].copy()
    d[DOAN] = d[DOAN].map(chuan_doan_duong)
    bc.ghi('B4_doan_duong_gop_cach_viet', so_dong_khac(truoc, d[DOAN]), DOAN)
    bc.kiem('B4_doan_duong_so_gia_tri', d[DOAN].nunique())

## B4.4 · Yếu tố khác

In [ ]:
def b4_yeu_to_khac(d, bc):
    """Ô chỉ chứa số (0, hoặc số tiền nhập nhầm cột) -> NaN."""
    la_so = d[YTK].fillna('').astype(str).str.match(r'^[\d.,\s]+$') & d[YTK].notna()
    d.loc[la_so, YTK] = np.nan
    bc.ghi('B4_yeu_to_khac_so_nan', la_so.sum(), YTK)

## B4.5 · Hình dáng

In [ ]:
SUA_CHINH_TA = {'cân đốicân đối': 'cân đối', 'kha scân đối': 'khá cân đối', 'cân dối': 'cân đối',
                'câm đối': 'cân đối', 'cân đôi': 'cân đối', 'không ân đối': 'không cân đối',
                'vạc góc': 'vạt góc', 'phúc tạp': 'phức tạp', 'hinh ': 'hình '}
KHONG_PHAI_HINH_DANG = {'không', 'khá', 'nhà hẻm'}
NHAN_GOC = r'(rất không cân đối|không cân đối|khá cân đối|cân đối|nở hậu|tóp hậu|hình chữ l)'


def sua_chinh_ta(x):
    if pd.isna(x):
        return x
    x = re.sub(r'^\d+(?=[^\W\d_])', '', x)          # số dính đầu chữ: 1cân đối
    x = re.sub(r'(?<=[^\W\d_])\d+$', '', x)         # số dính cuối chữ: cân đối5
    for sai, dung in SUA_CHINH_TA.items():
        x = x.replace(sai, dung)
    return 'cân đối' if x == 'đối' else x


def thong_nhat_hinh_dang(x):
    if pd.isna(x):
        return x
    x = re.sub(r'\bhình chữ l\b|\bchữ l\b|\bhình l\b', 'hình chữ l', x)
    x = x.replace('(l)', '(hình chữ l)')
    x = re.sub(r'^' + NHAN_GOC + r'\s*[-,]\s*(.+)$', lambda m: f'{m.group(1)}; {m.group(2).strip()}', x)
    x = re.sub(r'\s*\(\s*([^()]*?)\s*\)\s*', r'; \1; ', x)      # 'a (b)' -> 'a; b'
    x = re.sub(r'\s+', ' ', x)
    x = re.sub(r'\s*;\s*(?:;\s*)*', '; ', x)
    return x.strip(' ;')


def b4_hinh_dang(d, bc):
    """Ghi kết quả vào cột mới `hinh_dang_sach`; cột Hình dáng giữ nguyên."""
    hd = d[HD].copy()

    # (1) số hoặc một chữ cái lẻ -> NaN
    s = hd.fillna('')
    la_so = (s.str.match(r'^[\d.,\s]+$') | s.str.match(r'^[a-zà-ỹ]$')) & hd.notna()
    bc.ghi('B4_hinh_dang_so_nan', la_so.sum(), 'hinh_dang_sach')
    hd[la_so] = np.nan

    # (2) sửa lỗi chính tả, nhãn không phải hình dáng -> NaN
    truoc = hd.copy()
    hd = hd.map(sua_chinh_ta)
    khong_phai = hd.isin(KHONG_PHAI_HINH_DANG)
    hd[khong_phai] = np.nan
    n_sua = ((truoc.fillna('') != hd.fillna('')) & hd.notna()).sum()
    bc.ghi('B4_hinh_dang_sua_chinh_ta', n_sua, 'hinh_dang_sach')
    bc.ghi('B4_hinh_dang_khong_phai_hinh_dang_nan', khong_phai.sum(), 'hinh_dang_sach')

    # (3) thống nhất cách viết, các ý cách nhau bằng '; '
    truoc = hd.copy()
    hd = hd.map(thong_nhat_hinh_dang)
    n_tn = ((truoc.fillna('') != hd.fillna('')) & hd.notna()).sum()
    bc.ghi('B4_hinh_dang_thong_nhat_cach_viet', n_tn, 'hinh_dang_sach')

    d['hinh_dang_sach'] = hd
    bc.kiem('B4_hinh_dang_so_nhan', hd.nunique())
    bc.kiem('B4_hinh_dang_nan', hd.isna().sum())

## B4.6 · Chi tiết: tách thành các ý cách nhau bằng '; '

In [ ]:
MO_TA = (r'\b(?:nhà hoàn thiện|nhà nát|nhà thô|nhà cũ|nhà trọ|dãy trọ|tòa nhà|nhà xưởng|xưởng|khách sạn|chdv|'
         r'đầy đủ nội thất|có nội thất|nội thất|có thang máy|có hầm|đất trống|đất nền|biệt thự|căn hộ|nhà phố|nhà cấp 4|cấp 4|c4|'
         r'bê tông|cùng khu|góc \d+|cao cấp|hoàn thiện|cũ)\b')
MO_TA_DAU = r'^(?:nhà|đất|trọ|nền)\b\s*'
SO_NHA_AB = r'(?<!đường )(?<!kdc )(?<!khu phố )(?<!khu tái định cư )\b(?:số )?\d+[a-z]?(?:\s*/\s*\d*[a-z]?)+'
SO_NHA_DON = r'^(?:số |nhà )?\d{1,4}[a-z]?(?:\s*-\s*\d{1,4}[a-z]?)?$'
MT_CHU = r'\b\d?mt\b|mặt tiền|shophouse|\bmtnb\b'
HEM = r'\bhẻm\b'
THU_TU_Y = [r'^đường\b', r'^kdc\b', r'^khu\b', r'^ấp\b']   # thứ tự các ý sau ý vị trí


def _chuan_viet_tat(s):
    s = re.sub(r'https?://\S+', ' ', s)
    s = re.sub(r'\bnahf\b', 'nhà', s)
    s = re.sub(r'\bkhu dân cư\b|\bkđc\b|\bkhu dc\b', 'kdc', s)
    s = re.sub(r'(?<![\w-])(?:khu )?tđc\b|\bktđc\b', 'khu tái định cư', s)
    s = re.sub(r'\bkp\b\s*', 'khu phố ', s)
    s = re.sub(r'\bkno\b', 'khu nhà ở', s)
    s = re.sub(r'\bđnb\b', 'đường nội bộ', s)
    s = re.sub(r'\bđs\s*(?=\d)', 'đường số ', s)
    s = re.sub(r'\bmt nb\b', 'mtnb', s)
    s = re.sub(r'\b(tl|ql|dt|đt|hl|tx|ht|đht|ta|tch|tth) (\d+)\b', r'\1\2', s)
    s = re.sub(r'(?<!đường )\bnội bộ\b', 'đường nội bộ', s)
    s = re.sub(r'(^|[,;(]\s*|\b(?:mt|mtnb|hẻm|nhà|đất)\s+)đường (\d+[a-z]?)(?=\s*(?:$|[,;)]|-|\s))', r'\1đường số \2', s)
    # đánh dấu chỗ tách: luôn trước 'kdc'; trước 'đường'/'khu' chỉ khi chúng mở đầu cụm mới
    s = re.sub(r'\s+(?=kdc\b)', ' | ', s)
    s = re.sub(r'\b(mt|mtnb|hẻm|nhà|đất|bộ|\d+[a-z]?)\s+(?=(?:đường|khu)\b)', r'\1 | ', s)
    s = re.sub(r'\s+-\s+(?=(?:kdc|khu|đường|hẻm)\b)', ' | ', s)
    return re.sub(r'\s+', ' ', s).strip()


def _thu_tu(y):
    for i, p in enumerate(THU_TU_Y):
        if re.match(p, y):
            return i + 1
    return len(THU_TU_Y) + 1


def _don_dau(t):
    """Bỏ khoảng trắng thừa, dấu câu ở hai đầu và ngoặc lẻ."""
    t = re.sub(r'\s+', ' ', t)
    t = re.sub(r'\(\s+', '(', t)
    t = re.sub(r'\s+\)', ')', t).strip(' -,')
    while t.startswith('(') and t.count('(') > t.count(')'):
        t = t[1:].strip()
    while t.endswith(')') and t.count(')') > t.count('('):
        t = t[:-1].strip()
    if t.startswith('(') and t.endswith(')') and t.count('(') == 1:
        t = t[1:-1].strip()
    return t.strip(' -,')


def _bo_mo_ta_dau(t):
    while re.match(MO_TA_DAU, t):
        t = _don_dau(re.sub(MO_TA_DAU, '', t))
    return t


def _bo_chu_duong(t):
    """'đường huỳnh thúc kháng' -> 'huỳnh thúc kháng'; giữ 'đường số 7', 'đường nội bộ', 'đường d1'."""
    m = re.match(r'^đường (.+)$', t)
    if not m:
        return t
    sau = m.group(1)
    if re.match(r'^(?:số|nội bộ)\b', sau) or ' ' not in sau:
        return t
    return sau


def tach_chi_tiet(x, duong_pho=None):
    """Chuỗi các ý cách nhau bằng '; ': vị trí (mt, hẻm) trước, rồi đường, kdc, khu, ấp, phần còn lại."""
    if pd.isna(x):
        return np.nan
    s = _chuan_viet_tat(str(x))
    if not s:
        return np.nan
    duong = '' if duong_pho is None or pd.isna(duong_pho) else str(duong_pho)
    co_mt_ca_cau = bool(re.search(MT_CHU, s))
    s = re.sub(MO_TA, ' ', s)
    s = re.sub(r'\(\s*\)', ' ', s)
    manh = [p for p in re.split(r'[;,|]', s) if p.strip(' -()')]

    vi_tri, con = [], []
    for i, m in enumerate(manh):
        m = _bo_mo_ta_dau(_don_dau(m))
        if i == 0 and re.match(SO_NHA_DON, m):              # mảnh đầu chỉ là số nhà -> mặt tiền
            vi_tri.append('mt')
            continue
        if re.search(r'\bmtnb\b', m):
            vi_tri.append('mt')
            con.append('đường nội bộ')
        co_mt = bool(re.search(MT_CHU, m))
        co_hem = bool(re.search(HEM, m))
        if co_mt:
            vi_tri.append('mt')
        if co_hem:
            vi_tri.append('hẻm')
        if re.search(SO_NHA_AB, m) and not co_mt_ca_cau:   # số nhà dạng 57/1a -> hẻm
            vi_tri.append('hẻm')
        m = re.sub(SO_NHA_AB, ' ', m)
        m = re.sub(MT_CHU + '|' + HEM, ' ', m)
        m = re.sub(r'^(?:số |nhà )?\d{1,4}[a-z]?\b', ' ', m.strip())
        m = _bo_chu_duong(_bo_mo_ta_dau(_don_dau(m)))
        if (co_mt or co_hem) and duong and m and (m in duong or duong in m):
            m = ''                                          # tên đường trùng cột Đường phố
        if m and m not in ('nhà', 'đất', 'số'):
            con.append(m)

    y = [v for v in ('mt', 'hẻm') if v in vi_tri] + sorted(dict.fromkeys(con), key=_thu_tu)
    y = list(dict.fromkeys(y))
    return '; '.join(y) if y else np.nan


def b4_chi_tiet(d, bc):
    """Bản chữ trước khi tách được giữ trong `_chi_tiet_b1` cho B5 (cờ) và B7 (so trùng)."""
    d['_chi_tiet_b1'] = d[CT]
    d[CT] = [tach_chi_tiet(a, b) for a, b in zip(d['_chi_tiet_b1'], d[DUONG])]
    bc.ghi('B4_chi_tiet_tach_y', so_dong_khac(d['_chi_tiet_b1'], d[CT]), CT)
    bc.kiem('B4_chi_tiet_so_gia_tri', d[CT].nunique())
    bc.kiem('B4_chi_tiet_nan', d[CT].isna().sum())

## B5 · Mâu thuẫn VT – KC – Chi tiết

In [ ]:
def b5_co_nhan_dien(d, bc):
    """Bốn cờ đọc trên Chi tiết trước khi tách ý."""
    ct = d['_chi_tiet_b1'].fillna('')
    d['la_hem'] = ct.str.contains('hẻm')
    d['co_mat_tien'] = ct.str.contains(r'\bmt\b|mặt tiền|shophouse', regex=True)
    d['chi_la_mat_tien'] = ct.str.match(r'^(nhà |đất )?(mt|mặt tiền)$')
    d['la_so_nha'] = ct.str.match(r'^(số )?\d{1,4} ?[a-z]?$')
    d['co_bang_chung_mat_tien'] = d['co_mat_tien'] | d['la_so_nha']
    for c in ['la_hem', 'co_mat_tien', 'chi_la_mat_tien', 'la_so_nha']:
        bc.kiem('B5_' + c, d[c].sum())


def b5_sua_mat_tien(d, bc):
    """Sửa 1: Chi tiết chỉ ghi 'mt' -> VT = vt1, KC = 0 (trừ vt2–vt4 có KC > 0: chỉ gắn cờ)."""
    chi_gan_co = d['chi_la_mat_tien'] & d[VT].isin(VT234) & (d[KC] > 0)
    ap_dung = d['chi_la_mat_tien'] & ~chi_gan_co
    sua_vt = ap_dung & (d[VT] != 'vt1')
    sua_kc = ap_dung & (d[KC] > 0)
    d['da_sua_vt_theo_mat_tien'] = sua_vt | sua_kc
    d.loc[ap_dung, VT] = 'vt1'
    d.loc[ap_dung, KC] = 0
    bc.ghi('B5_sua1_vt', sua_vt.sum(), VT)
    bc.ghi('B5_sua1_kc', sua_kc.sum(), KC)
    bc.kiem('B5_sua1_chi_gan_co', chi_gan_co.sum())


def b5_sua_kc_0(d, bc):
    """Sửa 2: VT khác vt1 có KC = 0 mà không có bằng chứng mặt tiền -> KC = NaN."""
    nan_kc = (d[VT] != 'vt1') & (d[KC] == 0) & ~d['co_bang_chung_mat_tien']
    d['kc_thieu_du_lieu'] = nan_kc
    d.loc[nan_kc, KC] = np.nan
    bc.ghi('B5_sua2_kc_nan', nan_kc.sum(), KC)


def b5_canh_bao(d, bc):
    d['canh_bao_vt1'] = (d[VT] == 'vt1') & (d['la_hem'] | (d[KC] > 0) | (d[NGO] < 4))
    d['canh_bao_mt_vt234'] = d['co_mat_tien'] & d[VT].isin(VT234) & (d[KC] > 0)
    bc.ghi('B5_canh_bao_vt1', d['canh_bao_vt1'].sum(), 'chỉ gắn cờ, không sửa')
    bc.ghi('B5_canh_bao_mt_vt234', d['canh_bao_mt_vt234'].sum(), 'chỉ gắn cờ, không sửa')

## B6 · Ô kích thước và cột tỷ lệ

In [ ]:
def b6_kich_thuoc(d, bc):
    """Sửa 3: MT == DT -> MT = DT / CD; CD == DT -> CD = DT / MT. Diện tích giữ nguyên."""
    mt_sai = d[MT] == d[DT]
    cd_sai = d[CD] == d[DT]
    d['da_sua_kich_thuoc'] = mt_sai | cd_sai
    d.loc[mt_sai, MT] = d.loc[mt_sai, DT] / d.loc[mt_sai, CD]
    d.loc[cd_sai, CD] = d.loc[cd_sai, DT] / d.loc[cd_sai, MT]
    bc.ghi('B6_sua3_mt_bang_dt', mt_sai.sum(), MT)
    bc.ghi('B6_sua3_cd_bang_dt', cd_sai.sum(), CD)

    d['ty_le_kt_dt'] = d[MT] * d[CD] / d[DT]
    d['ty_le_dai_rong'] = d[CD] / d[MT]

## B8 · Ba cột chỉ số theo rule của công ty, tính trên giá trị gốc

In [ ]:
CAN_TREN_NGO = [1, 1.5, 2, 2.5, 3, 3.5, 4, 5, 7, 10, 15, 25]     # m, mức 1..13
CAN_TREN_DIEN_TICH = [30, 50, 70, 100, 150, 200, 300]           # m², mức 1..8


def chia_muc(x, can_tren):
    """Đổi giá trị số thành mức 1..n theo danh sách cận trên (≤ cận trên); NaN giữ NaN."""
    v = x.to_numpy(dtype=float)
    muc = np.searchsorted(np.array(can_tren, dtype=float), v, side='left') + 1
    return pd.Series(np.where(np.isnan(v), np.nan, muc), index=x.index).astype('Int64')


def b8_chi_so_ngo_dien_tich(d, bc):
    d[CS_NGO] = chia_muc(d['_do_rong_goc'], CAN_TREN_NGO)
    d[CS_DT] = chia_muc(d['_dien_tich_goc'], CAN_TREN_DIEN_TICH)
    for cot, cty in [(CS_NGO, '_cs_ngo_cty'), (CS_DT, '_cs_dt_cty')]:
        khac = (d[cot].astype(float) != d[cty].astype(float)).sum()
        bc.ghi(f'B8_{cot.lower()}_khac_cong_ty', khac, cot)


def _mode_xac_dinh(s):
    """Giá trị xuất hiện nhiều nhất; hoà thì lấy giá trị nhỏ hơn."""
    vc = s.value_counts().sort_index().sort_values(ascending=False, kind='stable')
    return vc.index[0]


def dung_bang_tra_hinh_dang(raw):
    """Với mỗi chuỗi Hình dáng nguyên văn: hệ số công ty gán nhiều nhất, ưu tiên các dòng từ NGAY_BANG_HIEN_HANH."""
    goc = pd.DataFrame({'hinh_dang_nguyen_van': raw[HD], 'he_so': raw[CS_HD].round(3), 'ngay': doc_ngay(raw[NGAY])})
    hien_hanh = goc['ngay'] >= NGAY_BANG_HIEN_HANH
    bang = (goc.groupby('hinh_dang_nguyen_van')['he_so']
               .agg(he_so_ca_giai_doan=_mode_xac_dinh, so_dong_ca_giai_doan='size')
               .join(goc[hien_hanh].groupby('hinh_dang_nguyen_van')['he_so']
                       .agg(he_so_hien_hanh=_mode_xac_dinh, so_dong_hien_hanh='size'), how='left'))
    bang['he_so'] = bang['he_so_hien_hanh'].fillna(bang['he_so_ca_giai_doan'])
    bang['nguon_he_so'] = np.where(bang['he_so_hien_hanh'].notna(), 'từ ngày hiện hành', 'cả giai đoạn')
    bang['so_dong_hien_hanh'] = bang['so_dong_hien_hanh'].fillna(0).astype(int)
    return bang


def luu_bang_tra(bang, path):
    # QUOTE_NONNUMERIC giữ nguyên khoảng trắng đầu/cuối của chuỗi Hình dáng
    bang.reset_index().to_csv(path, index=False, encoding='utf-8-sig', quoting=csv.QUOTE_NONNUMERIC)


def doc_bang_tra(path):
    # keep_default_na=False: chuỗi như 'NA', 'null' vẫn là khoá tra hợp lệ
    bang = pd.read_csv(path, keep_default_na=False, dtype={'hinh_dang_nguyen_van': str})
    bang = bang.set_index('hinh_dang_nguyen_van')
    bang['he_so'] = bang['he_so'].astype(float)
    return bang


def b8_chi_so_hinh_dang(d, bang, bc):
    """Tra nguyên văn chuỗi Hình dáng; không tra được thì gán HE_SO_MAC_DINH."""
    d[CS_HD] = d['_hinh_dang_nguyen_van'].map(bang['he_so']).fillna(HE_SO_MAC_DINH)
    khac = (d[CS_HD].round(3) != d['_cs_hd_cty']).sum()
    bc.ghi('B8_chi_so_hinhdang_khac_cong_ty', khac, CS_HD)
    print(f'  khớp cột của công ty: {(1 - khac / len(d)) * 100:.2f}% số dòng')

## B9 · Kiểm thử đầu ra

In [ ]:
def b9_kiem_thu(d, bc):
    """Kiểm các điều kiện của dữ liệu sạch, trả về bảng đầu ra (đã bỏ mọi cột phụ)."""
    out = d[cot_xuat(d)].copy()
    bc.kiem('B9_so_dong', len(out))
    bc.kiem('B9_so_cot', out.shape[1])
    bc.kiem('B9_duong_pho_nan', out[DUONG].isna().sum())

    dk('không còn dòng trùng trên khoá B7', not d.duplicated(subset=KHOA_B7).any())
    text_out = [c for c in out.columns if not pd.api.types.is_numeric_dtype(out[c])]
    dk('không còn chữ hoa ở mọi cột chữ',
       all((out[c].dropna().astype(str) == out[c].dropna().astype(str).str.lower()).all() for c in text_out))
    dk('VT chỉ còn vt1–vt4', set(out[VT].unique()) <= {'vt1', 'vt2', 'vt3', 'vt4'})
    dk('số mặt tiền chỉ 1–4 hoặc NaN', out[SMT].dropna().isin([1, 2, 3, 4]).all())
    dk('không còn 0 ở độ rộng và mặt tiền', not (out[NGO] == 0).any() and not (out[MT] == 0).any())
    dk('KC không vượt 2.500', (out[KC].dropna() <= 2500).all())
    dk('chi_la_mat_tien đúng rule (trừ dòng có canh_bao_mt_vt234)',
       not (out['chi_la_mat_tien'] & ~out['canh_bao_mt_vt234'] & ((out[VT] != 'vt1') | (out[KC] > 0))).any())
    dk('mọi dòng VT ≠ vt1 có KC = 0 đều có bằng chứng mặt tiền',
       not ((out[VT] != 'vt1') & (out[KC] == 0) & ~out['co_bang_chung_mat_tien']).any())
    dk('Chi tiết đúng định dạng ý; ý',
       out[CT].dropna().map(lambda s: all(t and t.strip() == t for t in s.split('; '))).all())
    dk('target > 0', (out[TARGET] > 0).all())
    dk('cột tỷ lệ không có inf', not np.isinf(out[['ty_le_kt_dt', 'ty_le_dai_rong']]).any().any())
    dk('Chi_so_ngo chỉ nhận 1–13, không NaN', out[CS_NGO].between(1, 13).all())
    dk('Chi_so_dientich chỉ nhận 1–8, không NaN', out[CS_DT].between(1, 8).all())
    dk('Chi_so_hinhdang không NaN và nằm trong 0,70–1,05', out[CS_HD].between(0.70, 1.05).all())

    so_dong_sua = (out['da_sua_vt_theo_mat_tien'] | out['kc_thieu_du_lieu'] | out['da_sua_kich_thuoc']).sum()
    bc.kiem('B9_so_dong_bi_sua', so_dong_sua)
    print(f'Dòng bị sửa: {so_dong_sua} ({so_dong_sua / len(out) * 100:.2f}%)')
    return out